# Class 10 — All subjects & between-subject variance

A single subject can't tell you whether a decoder generalises — BCI has a well-documented between-subject variability problem. This notebook repeats the CSP+LDA decoder across all 9 subjects of BNCI2014_001 and reports the population-level mean and standard deviation, not a single number.

## Setup

In [1]:
import numpy as np
import pandas as pd
import torch
import mne
import matplotlib.pyplot as plt

from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, GroupKFold

from mne.decoding import CSP

from pyriemann.estimation import Covariances
from pyriemann.classification import MDM, TSClassifier

mne.set_log_level("WARNING")
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


/Users/vicky/Documents/motor_imagery_decoding/motor-imagery-decoding/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
paradigm = MotorImagery(n_classes=4, fmin=8, fmax=30, tmin=0.5, tmax=3.5)

csp_lda = Pipeline([
    ("CSP", CSP(n_components=6, reg="ledoit_wolf", log=True, norm_trace=False)),
    ("LDA", LinearDiscriminantAnalysis()),
])


Choosing from all possible events


## Loop over all 9 subjects

In [3]:
all_results = {}

for subject in range(1, 10):
    dataset = BNCI2014_001()
    dataset.subject_list = [subject]

    X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[subject])

    groups = metadata["session"]
    cv = GroupKFold(n_splits=2)

    scores = cross_val_score(csp_lda, X, y, cv=cv, groups=groups)
    all_results[subject] = scores.mean()
    print(f"Subject {subject}: {scores.mean():.3f}")


Subject 1: 0.769
Subject 2: 0.531
Subject 3: 0.780
Subject 4: 0.559
Subject 5: 0.396
Subject 6: 0.467
Subject 7: 0.590
Subject 8: 0.726
Subject 9: 0.653


## Population-level result

In [4]:
values = list(all_results.values())
mean_acc = np.mean(values)
std_acc = np.std(values)
print(f"Mean: {mean_acc:.3f}, Std: {std_acc:.3f}")


Mean: 0.608, Std: 0.127


**Result:** Mean **0.608**, Std **0.127** across 9 subjects — well above chance, but with substantial between-subject variability. Subjects 3, 1 and 8 respond well to the paradigm; subjects 5 and 6 sit close to the usability floor, consistent with the known "BCI-illiteracy" phenomenon.